In [46]:
from pyspark.sql import SparkSession
from pyspark.sql.types import ArrayType, StringType, IntegerType
from pyspark.sql.functions import pandas_udf, lower, rand

from pyspark.ml.feature import CountVectorizer

import spacy

import pandas as pd

In [47]:
def wrapper_word_splitter(lang_model:str) -> callable:
    '''This function returns a pyspark UDF already set to split text to a list of words

    wrapper_word_splitter uses spacy inside; therefore, a name of a downloaded 
    model must be provided according to the target language
    '''
    nlp = spacy.load(lang_model, disable=["parser", "ner", "lemmatizer"])

    @pandas_udf(ArrayType(StringType()))
    def spacy_word_splitter_udf(batch):

        results = []
        for doc in nlp.pipe(batch.astype(str)):
            results.append([token.text for token in doc])

        return pd.Series(results)

    return spacy_word_splitter_udf


def wrapper_word2index(spark, vocab_index:dict, unk_word:str = "[UNK]", add_start:str|None = None, add_end:str|None = None) -> callable:
    '''Thus function returns a pyspark UDF to convert a list of words found
    in a pyspark.sql.DataFrame into a list of index (tokens)

    Parameters
    ----------

    spark: pyspark.sql.SparkSession
        Spark session to broadcast vocab

    vocav_index: dict
        A dict containing the vocab and index to map

    unk_word: str
        Name of the string describing the unknown token expected to be found in vocab_index

    add_start:str|None = None
        Wheather or not add the start token in the sentence. if not null, a string is expected to be found in vocab_index

    add_end:str|None = None
        Wheather or not add the end token in the sentence. if not null, a string is expected to be found in vocab_index    
    '''

    broadcast_vocab = spark.sparkContext.broadcast(vocab_index)

    token_unk = vocab_index[unk_word]
    token_sos = [vocab_index[add_start]] if add_start else []
    token_end = [vocab_index[add_end]]  if add_end  else []

    @pandas_udf(ArrayType(IntegerType()))
    def tokenizer_word2index_udf(batch):

        vocab_index = broadcast_vocab.value

        def tokenize(array_words):
            return token_sos + [vocab_index.get(w, token_unk) for w in array_words] + token_end

        return batch.apply(tokenize)

    return tokenizer_word2index_udf

In [48]:
spark = (
    SparkSession.
    builder.
    appName("Preprocess_spa_eng_dataset").
    master("local[*]").
    getOrCreate()
)

spark

In [49]:
raw_spa_eng_txt = (
    spark.
    read.
    option("delimiter", "\t").
    option("header", None).
    csv("../data/raw/spa-eng/")
)

raw_spa_eng_txt = raw_spa_eng_txt.select(
    lower("_c0").alias("eng"),
    lower("_c1").alias("spa")
)
raw_spa_eng_txt.show(5)

+---+-------+
|eng|    spa|
+---+-------+
|go.|    ve.|
|go.|  vete.|
|go.|  vaya.|
|go.|váyase.|
|go.|    id.|
+---+-------+
only showing top 5 rows



In [50]:
splitter_eng = wrapper_word_splitter("en_core_web_sm")
splitter_spa = wrapper_word_splitter("es_core_news_sm")

splitted_spa_eng = (
    raw_spa_eng_txt.
    select(
        splitter_eng("eng").alias("eng_arr"),
        splitter_spa("spa").alias("spa_arr")
    )
)

In [51]:
count_vec_eng = CountVectorizer(inputCol="eng_arr", outputCol="features")
count_vec_spa = CountVectorizer(inputCol="spa_arr", outputCol="features")

cv_model_eng = count_vec_eng.fit(splitted_spa_eng)
cv_model_spa = count_vec_spa.fit(splitted_spa_eng)

In [52]:
special_tokens = ["[PAD]", "[START]", "[END]", "[UNK]"]
spa_vocab = special_tokens + cv_model_spa.vocabulary
eng_vocab = special_tokens + cv_model_eng.vocabulary

spa_word2index = {word:idx for idx, word in enumerate(spa_vocab)}
eng_word2index = {word:idx for idx, word in enumerate(eng_vocab)}

In [53]:
# Input should contain end token
# Ouput must contain start token
# Target must contain end token
tokenizer_eng_input  = wrapper_word2index(spark, eng_word2index, add_end="[END]")
tokanizer_spa_output = wrapper_word2index(spark, spa_word2index, add_start="[START]")
tokanizer_spa_target = wrapper_word2index(spark, spa_word2index, add_end="[END]")

tokenized_spa_eng = (
    splitted_spa_eng.
    select(
        tokenizer_eng_input("eng_arr").alias("eng_tokens_in"),
        tokanizer_spa_output("spa_arr").alias("spa_tokens_out"),
        tokanizer_spa_target("spa_arr").alias("spa_tokens_target"),
    )
)

In [54]:
tokenized_spa_eng.show()

+--------------+------------------+------------------+
| eng_tokens_in|    spa_tokens_out| spa_tokens_target|
+--------------+------------------+------------------+
|    [49, 4, 2]|       [1, 388, 4]|       [388, 4, 2]|
|    [49, 4, 2]|      [1, 1392, 4]|      [1392, 4, 2]|
|    [49, 4, 2]|       [1, 492, 4]|       [492, 4, 2]|
|    [49, 4, 2]|      [1, 5705, 4]|      [5705, 4, 2]|
|    [49, 4, 2]|      [1, 4555, 4]|      [4555, 4, 2]|
|    [49, 4, 2]|      [1, 1961, 4]|      [1961, 4, 2]|
|    [49, 4, 2]|      [1, 6422, 4]|      [6422, 4, 2]|
|  [1883, 4, 2]|      [1, 1752, 4]|      [1752, 4, 2]|
| [436, 122, 2]| [1, 82, 1663, 81]| [82, 1663, 81, 2]|
| [436, 122, 2]| [1, 82, 8980, 81]| [82, 8980, 81, 2]|
| [436, 122, 2]|[1, 82, 16662, 81]|[82, 16662, 81, 2]|
| [436, 122, 2]| [1, 82, 8143, 81]| [82, 8143, 81, 2]|
| [436, 122, 2]|[1, 82, 10027, 81]|[82, 10027, 81, 2]|
|   [436, 4, 2]|      [1, 8143, 4]|      [8143, 4, 2]|
|   [436, 4, 2]|     [1, 10027, 4]|     [10027, 4, 2]|
|   [81, 1

In [55]:
tokenized_spa_eng_shuffle = (
    tokenized_spa_eng.
    withColumn("rand_col", rand(seed=42)).
    orderBy("rand_col").
    drop("rand_col")
)

tokenized_spa_eng_shuffle.cache()

train, test = tokenized_spa_eng_shuffle.randomSplit([0.8, 0.2], seed=42)

In [56]:
(
    train.
    coalesce(4).
    write.
    mode("overwrite").
    parquet("../data/main/spa-eng/train_spa_eng.parquet")
)
(
    test.
    coalesce(4).
    write.
    mode("overwrite").
    parquet("../data/main/spa-eng/test_spa_eng.parquet")
)